In [19]:
from embedder import Embedder

embed = Embedder()

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

In [20]:
q = 'How does approximate nearest neighbor search work?'
v = embed.encode(q)

print(len(v))
print(f'{v[0]:.2f}')
v[0]

384
-0.02


np.float64(-0.02058203437252893)

In [21]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [22]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [23]:
target_doc = {}
target_file = "02-vector-search/lessons/07-sqlitesearch-vector.md"

for doc in documents:
    if doc['filename'] == target_file:
        target_doc = doc
        break

print(target_doc)

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [24]:
q_target = target_doc['content']
v_target = embed.encode(q_target)

v.dot(v_target)

np.float64(0.36107026789538205)

In [26]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))
texts = []

for chunk in chunks:
    texts.append(chunk['content'])

X = embed.encode_batch(texts)
X.shape

scores = X.dot(v)
scores

295


array([ 3.15187705e-01,  2.01479593e-01,  5.90559560e-02, -7.67661858e-02,
        1.18452480e-01, -1.41697042e-01, -2.81406552e-02, -4.65669225e-02,
       -2.06994704e-02, -6.07744087e-02,  2.13273853e-01,  8.87601799e-02,
       -1.97269351e-02,  3.11630016e-01,  5.51034674e-01,  2.04008048e-01,
        2.12515842e-01,  1.93649180e-01,  2.51961293e-01,  1.31078643e-01,
        2.59120579e-01,  7.63816008e-02,  9.59193707e-02,  9.81472975e-03,
       -3.59106882e-02,  1.01211577e-02,  1.10786937e-01, -9.90259208e-02,
       -3.71170151e-02,  7.59057570e-02, -3.35340540e-02,  8.86841309e-03,
        1.02636405e-01,  6.89614876e-02,  1.29408856e-01,  2.57709091e-01,
        3.23680614e-01,  1.06350075e-01,  5.61891367e-02,  2.34017457e-01,
        1.97954387e-01,  9.64296290e-02,  1.93709917e-01,  2.16719271e-01,
        3.48340456e-01,  5.10906092e-02,  2.05212837e-01,  1.05416170e-01,
       -3.25432514e-02,  4.94665548e-02,  2.38574865e-01, -3.44207108e-02,
        1.82165438e-01,  

In [27]:
idx_max = scores.argmax()

idx_max, chunks[idx_max]['filename']

(np.int64(94), '02-vector-search/lessons/07-sqlitesearch-vector.md')

In [28]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['filename'])

vindex.fit(X, chunks)

In [29]:
query = 'What metric do we use to evaluate a search engine?'
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

In [30]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

In [31]:
q5 = "How do I store vectors in PostgreSQL?"
v_q5 = embed.encode(q5)

In [32]:
text_results = index.search(q5, num_results=5)
text_results

[{'start': 4000,
  'content': 'get 0.01.\n\nThe first score for `q1` vs `d` (0.32) is higher, so that query is more\nsimilar to the document about registration. The second score for `q2`\nvs `d` sits near 0, because installing Docker has nothing to do with\nregistration. A score near 0 means the two vectors are about as\ndifferent as they can be.\n\nThat\'s the whole idea behind vector search: similar texts get similar\nvectors, and a dot product tells us how similar.\n\n## Cosine similarity\n\nThe `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with\nunit length. When both vectors are normalized, the dot product equals\ncosine similarity. That\'s why the model documentation says it "uses\ncosine similarity."\n\nCosine similarity measures the angle between two vectors, ignoring\ntheir length:\n\n- 1.0 = same direction (similar)\n- 0.0 = perpendicular (unrelated)\n- -1.0 = opposite direction (opposite meaning)\n\nFormally, if `theta` is the angle between two vectors, cosin

In [33]:
for result in index.search(q5, num_results=5):
    print(result['filename'])

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [34]:
vec_results = vindex.search(v_q5, num_results=5)
vec_results

[{'start': 0,
  'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWO

In [35]:
for result in vindex.search(v_q5, num_results=5):
    print(result['filename'])

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [36]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [37]:
query = "How do I give the model access to tools?"
v_query = embed.encode(query)

In [38]:
search_list = index.search(query, num_results=5)
vsearch_list = vindex.search(v_query, num_results=5)

In [39]:
rrf([search_list, vsearch_list], k=60, num_results=5)


[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 